# Part 1d. Random Classifier Performance

In [1]:
# Imports

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score

In [2]:
# Load the dataset

df = pd.read_csv("hf://datasets/processvenue/SARCASM_VS_NON_SARCASM/PROJ_SARCASM_VS_NON_SARCASM_6000_0004(HF).csv", index_col='S.No')

/Users/gal.lagelpibuxade/Desktop/sarcasm_text_classification/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Generate the labels from the annotators agreement

df['label'] = (df['Annotator 1']==df['Annotator 2'])
df['label'] = np.where(df['label'], df['Annotator 1'], 'Unknown')
#df.to_csv('../data/sarcasm_labelled.csv')

iaa = len(df[df['label']!='Unknown']) / len(df)
print(f"Inter-Annotator Agreement: {round(iaa, 4)*100}%")

df = df[df['label']!='Unknown']

df

Inter-Annotator Agreement: 63.92%


,Sentence,Annotator 1,Annotator 2,IAA,label
S.No,,,,,
1,with such eloquent and compelling arguments as...,Sarcastic,Sarcastic,Agree,Sarcastic
2,"fantasy,fairy tale, wild imagination, adsurd ,...",Non-Sarcastic,Non-Sarcastic,Agree,Non-Sarcastic
3,"'if carrying guns became legal, then police co...",Sarcastic,Sarcastic,Agree,Sarcastic
5,Poor little poopie. \r\nIf your gay sex agenda...,Sarcastic,Sarcastic,Agree,Sarcastic
6,":p I wish! If I did, I'd be as wealthy as the ...",Sarcastic,Sarcastic,Agree,Sarcastic
...,...,...,...,...,...
5994,"Nice try twisting my words, Simone. My post ab...",Sarcastic,Sarcastic,Agree,Sarcastic
5996,"Quit using movie stars. After Kerry lost, a lo...",Sarcastic,Sarcastic,Agree,Sarcastic
5997,Hoooahhhh! The smartest man in the world lives...,Sarcastic,Sarcastic,Agree,Sarcastic


If we apply a random model that 50% of the times attribute the sentence to Sarcastic label, and the other 50% of the times to the Non-Sarcastic label, if the panel was 100% balanced, so that half of the observations are of one type and the other half of the other type, we would have a performance of a 50% of accuracy, f1-score, precision, recall...

If the dataset is not perfectly balanced that doesen't apply. So let's first check the distribution of our database so that we can calculate the expected performance of a random classifier. 

In [4]:
df["label"].value_counts()

label
Sarcastic        2391
Non-Sarcastic    1406
Name: count, dtype: int64

We have 3797 observations, 2391 of them are Sarcastic (62,97%) and the other 1406 are Non-Sarcastic (37.03%). Based on this, if we apply a Random Classifier that 50% of the times says that the text is Sarcastic and the 50% restant diu que es Non-Sarcastic, this model will have as a performance:

We can assume that, considering Positive (1) = Sarcastic and Negative (0) = Non_Sarcastic, we have that:

- TP = 1195,5 (50% of the number of positives)
- FN = 1195,5 (50% of the number of positives)
- TN = 703 (50% of the number of negatives)
- FP = 703 (50% of the number of negatives)

Based on that, the expected performace of the different metrics are:

- Accuracy = $\frac{TP + TN}{Total} = \frac{1195,5 + 703}{3797} = 0,5$
- Recall (1) = $\frac{TP}{TP + FN} = \frac{1195,5}{1195,5 + 1195,5} = 0,5$
- Recall (0) = $\frac{TN}{TN + FP} = \frac{703}{703 + 703} = 0,5$
- Precision (1) = $\frac{TP}{TP + TN} = \frac{1195,5}{1195,5 + 703} = 0,6297$
- Precision (0) = $\frac{TN}{TN + TP} = \frac{703}{703 + 1195,5} = 0,3703$
- F1-score (1) = $2 * \frac{Precision (1) * Recall (1)}{Precision (1) + Recall (1)} = 2 * \frac{0,6297 * 0,5}{0,6297 + 0,5} = 0,56$
- F1-score (0) = $2 * \frac{Precision (0) * Recall (0)}{Precision (0) + Recall (0)} = 2 * \frac{0,3703 * 0,5}{0,3703 + 0,5} = 0,43$

Let's now contrast this results by actually implementing the random classifier:

In [5]:
from sklearn.metrics import (
    classification_report, 
    confusion_matrix,
    roc_curve, 
    auc, 
    precision_recall_curve, 
    average_precision_score
)

y_true = df['label'].values

np.random.seed(42)

# Random classifier that assigns 50% of the samples to 'Sarcastic' and 50% to 'Non-Sarcastic'
y_prob_random = np.random.choice([0.0, 1.0], size=len(y_true), p=[0.5, 0.5])
y_pred_random = np.where(y_prob_random == 1.0, 'Sarcastic', 'Non-Sarcastic')

print("\nRANDOM BENCHMARK PERFORMANCE:")
print(classification_report(y_true, y_pred_random))

macro_f1 = f1_score(y_true, y_pred_random, average='macro')
print(f"MACRO F1-SCORE: {macro_f1:.4f}")

print("\nCONFUSION MATRIX: ")
print(confusion_matrix(y_true, y_pred_random))


RANDOM BENCHMARK PERFORMANCE:
               precision    recall  f1-score   support

Non-Sarcastic       0.37      0.50      0.42      1406
    Sarcastic       0.63      0.50      0.56      2391

     accuracy                           0.50      3797
    macro avg       0.50      0.50      0.49      3797
 weighted avg       0.53      0.50      0.51      3797

MACRO F1-SCORE: 0.4906

CONFUSION MATRIX: 
[[ 696  710]
 [1190 1201]]


Most of the times, when the dataset is not balanced, we take as a Random classifier, one that respects the proportion of the classes in the datasets when doing the predictions (Stratified Random Classifier). If we implement this classifier, that 63% of the times is going to say that the observation is Sarcastic, and the other 37% of the times is going to say that is Non-Sarcastic, we can assume that:

- TP = 1505,6 (63% of the number of positives)
- FN = 885,4 (37% of the number of positives)
- TN = 520,6 (37% of the number of negatives)
- FP = 885,4 (63% of the number of negatives)

Based on that, the expected performace of the different metrics are:

- Accuracy = $\frac{TP + TN}{Total} = \frac{1505,6 + 520,6}{3797} = 0,53$
- Recall (1) = $\frac{TP}{TP + FN} = \frac{1505,6}{1505,6 + 885,4} = 0,63$
- Recall (0) = $\frac{TN}{TN + FP} = \frac{520,6}{520,6 + 885,4} = 0,37$
- Precision (1) = $\frac{TP}{TP + FP} = \frac{1505,6}{1505,6 + 885,4} = 0,63$
- Precision (0) = $\frac{TN}{TN + FN} = \frac{520,6}{520,6 + 885,4} = 0,37$
- F1-score (1) = $2 * \frac{Precision (1) * Recall (1)}{Precision (1) + Recall (1)} = 2 * \frac{0,63 * 0,63}{0,63 + 0,63} = 0,63$
- F1-score (0) = $2 * \frac{Precision (0) * Recall (0)}{Precision (0) + Recall (0)} = 2 * \frac{0,37 * 0,37}{0,37 + 0,37} = 0,37$

Let's now contrast this results by actually implementing the stratified random classifier:

In [6]:
# Stratified random classifier that assigns 63% of the samples to 'Sarcastic' and 37% to 'Non-Sarcastic'
np.random.seed(42)
y_prob_stratified = np.random.choice([0.0, 1.0], size=len(y_true), p=[0.37, 0.63])
y_pred_stratified = np.where(y_prob_stratified == 1.0, 'Sarcastic', 'Non-Sarcastic')

print("\nSTRATIFIED RANDOM BENCHMARK PERFORMANCE:")
print(classification_report(y_true, y_pred_stratified))

macro_f1 = f1_score(y_true, y_pred_stratified, average='macro')
print(f"MACRO F1-SCORE: {macro_f1:.4f}")

print("\nCONFUSION MATRIX: ")
print(confusion_matrix(y_true, y_pred_stratified))


STRATIFIED RANDOM BENCHMARK PERFORMANCE:
               precision    recall  f1-score   support

Non-Sarcastic       0.37      0.37      0.37      1406
    Sarcastic       0.63      0.63      0.63      2391

     accuracy                           0.53      3797
    macro avg       0.50      0.50      0.50      3797
 weighted avg       0.53      0.53      0.53      3797

MACRO F1-SCORE: 0.4985

CONFUSION MATRIX: 
[[ 522  884]
 [ 895 1496]]


Finally the last model that we are going to consider in this section is not a Random model itself, but it's very used as a benchmark when we have a imbalanced dataset. Is the model that always predicts the dominant class ('Sarcastic' in this case). In this case we will have:

- TP = 2391 (100% of the number of positives)
- FN = 0 (0% of the number of positives)
- TN = 0 (0% of the number of negatives)
- FP = 1406 (100% of the number of negatives)

Based on that, the expected performance of the different metrics are:

- Accuracy = $\frac{TP + TN}{Total} = \frac{2391 + 0}{3797} = 0.63$
- Recall (1) = $\frac{TP}{TP + FN} = \frac{2391}{2391 + 0} = 1.00$
- Recall (0) = $\frac{TN}{TN + FP} = \frac{0}{0 + 1406} = 0.00$
- Precision (1) = $\frac{TP}{TP + FP} = \frac{2391}{2391 + 1406} = 0.63$
- Precision (0) = $\frac{TN}{TN + FN} = \frac{0}{0 + 0} = 0.00$ *(Note: Technically undefined, defaults to 0)*
- F1-score (1) = $2 \times \frac{Precision(1) \times Recall(1)}{Precision(1) + Recall(1)} = 2 \times \frac{0.63 \times 1.00}{0.63 + 1.00} = 0.77$
- F1-score (0) = $2 \times \frac{Precision(0) \times Recall(0)}{Precision(0) + Recall(0)} = 2 \times \frac{0.00 \times 0.00}{0.00 + 0.00} = 0.00$

Let's now show these results by actually implementing the always 'Sarcastic' classifier:

In [7]:
# Classifier that always predicts the dominant class ('Sarcastic')

y_pred_dominant = np.full(len(y_true), 'Sarcastic')

print("\nALWAYS 'SARCASTIC' BENCHMARK PERFORMANCE:")
print(classification_report(y_true, y_pred_dominant, zero_division=0)) # zero_division=0 because Precision for 'Non-Sarcastic' is 0/0

macro_f1 = f1_score(y_true, y_pred_dominant, average='macro')
print(f"MACRO F1-SCORE: {macro_f1:.4f}")

print("\nCONFUSION MATRIX: ")
print(confusion_matrix(y_true, y_pred_dominant))


ALWAYS 'SARCASTIC' BENCHMARK PERFORMANCE:
               precision    recall  f1-score   support

Non-Sarcastic       0.00      0.00      0.00      1406
    Sarcastic       0.63      1.00      0.77      2391

     accuracy                           0.63      3797
    macro avg       0.31      0.50      0.39      3797
 weighted avg       0.40      0.63      0.49      3797

MACRO F1-SCORE: 0.3864

CONFUSION MATRIX: 
[[   0 1406]
 [   0 2391]]


The obtained metrics in the different models reflect the underlying logic of each baseline model. The purely Random model (50%-50%) results in a 50% recall for both categories and an overall accuracy of 50%, as it completely ignores the dataset's actual imbalance. The Stratified Random model, improves slightly the accuracy to 53% because it mimic the real 63%-37% class distribution, although its macro F1-score remains around 0.50 since it is still making uninformed guesses. Finally, the Always 'Sarcastic' model achieves the highest accuracy of all with a 63%, however, as expected, its macro metrics decreases to 0.39, because it entirely fails to identify the minority 'Non-Sarcastic' class. 

As we have explained during the EDA, because the imbalance in our dataset, the metric that we are going to use mainly to evaluate our models is the Macro-F1. Because of this, we are setting as a benchmark the random model with the higher score in it, the Stratified Random Model, with a Macro-F1 of 0.4985.